# Advanced Python Generators: Problems, Solutions, and Engineering Patterns

This notebook expands the supplied material on generator states into a comprehensive, executable problem set.

## Learning goals

By the end, you should be able to:

- reason precisely about `GEN_CREATED`, `GEN_RUNNING`, `GEN_SUSPENDED`, and `GEN_CLOSED`;
- predict where execution pauses around `yield`;
- capture a generator return value from `StopIteration`;
- use `send`, `throw`, and `close` safely;
- delegate with `yield from`;
- design lazy streaming algorithms and pipelines;
- inspect and debug suspended generators;
- guarantee cleanup with `try/finally`;
- test generators without accidentally consuming them;
- recognize re-entrancy, memory, and API-design pitfalls.

All examples use only the Python standard library.

## How to use this notebook

For each problem:

1. Read the specification and predict the behavior.
2. Write your own solution in a scratch cell.
3. Compare it with the supplied solution.
4. Run the assertions.
5. Modify the inputs and test edge cases.

### Best-practice checklist

- Keep generator functions focused on one streaming responsibility.
- Document whether a generator is one-shot, infinite, bidirectional, or resource-owning.
- Prefer `yield from` when true delegation is intended.
- Put cleanup in `finally`.
- Avoid raising `StopIteration` manually inside generator bodies.
- Test empty input, early termination, explicit `close()`, and exceptional paths.
- Do not convert to `list` unless materialization is required.

## 1. Core model: generator lifecycle

In [1]:
from inspect import getgeneratorstate
from typing import Any, Iterable, Iterator, TypeVar
from collections.abc import Generator

T = TypeVar("T")

def characters(text: str) -> Iterator[str]:
    for character in text:
        yield character

In [2]:
g = characters("abc")
assert getgeneratorstate(g) == "GEN_CREATED"

assert next(g) == "a"
assert getgeneratorstate(g) == "GEN_SUSPENDED"

assert list(g) == ["b", "c"]
assert getgeneratorstate(g) == "GEN_CLOSED"

### Example: observing `GEN_RUNNING`

A generator is running only while Python is actively executing its frame. To observe that state, inspect it from inside its own execution.

In [3]:
holder: dict[str, Iterator[str]] = {}

def self_reporting_generator() -> Iterator[str]:
    assert getgeneratorstate(holder["generator"]) == "GEN_RUNNING"
    yield "observed running state"

holder["generator"] = self_reporting_generator()
assert getgeneratorstate(holder["generator"]) == "GEN_CREATED"
assert next(holder["generator"]) == "observed running state"
assert getgeneratorstate(holder["generator"]) == "GEN_SUSPENDED"
holder["generator"].close()
assert getgeneratorstate(holder["generator"]) == "GEN_CLOSED"

## Problem 1 — Build a lifecycle tracer

Write `trace_lifecycle(generator)` that records:

- the initial state;
- the state after each successful `next`;
- every yielded value;
- the final state after exhaustion.

The helper must not leak `StopIteration`.

### Solution

In [4]:
def trace_lifecycle(generator: Iterator[T]) -> dict[str, list[object]]:
    states: list[str] = [getgeneratorstate(generator)]
    values: list[T] = []

    while True:
        try:
            values.append(next(generator))
            states.append(getgeneratorstate(generator))
        except StopIteration:
            states.append(getgeneratorstate(generator))
            break

    return {"states": states, "values": values}


trace = trace_lifecycle(characters("xyz"))
assert trace["values"] == ["x", "y", "z"]
assert trace["states"] == [
    "GEN_CREATED",
    "GEN_SUSPENDED",
    "GEN_SUSPENDED",
    "GEN_SUSPENDED",
    "GEN_CLOSED",
]
trace

{'states': ['GEN_CREATED',
  'GEN_SUSPENDED',
  'GEN_SUSPENDED',
  'GEN_SUSPENDED',
  'GEN_CLOSED'],
 'values': ['x', 'y', 'z']}

## Problem 2 — Prove evaluation order around `yield`

Create a generator that records events before computing a yielded value, during the computation, and immediately after the `yield`. Verify that the post-yield event occurs only on the next resume.

### Solution

In [5]:
events: list[str] = []

def expensive_square(number: int) -> int:
    events.append(f"compute:{number}")
    return number * number

def instrumented_squares(limit: int) -> Iterator[int]:
    for number in range(limit):
        events.append(f"before:{number}")
        yield expensive_square(number)
        events.append(f"after:{number}")

squares = instrumented_squares(2)

assert next(squares) == 0
assert events == ["before:0", "compute:0"]

assert next(squares) == 1
assert events == [
    "before:0",
    "compute:0",
    "after:0",
    "before:1",
    "compute:1",
]

try:
    next(squares)
except StopIteration:
    pass

assert events[-1] == "after:1"

## Problem 3 — Capture a generator's return value

A generator can `return value`. The value becomes `StopIteration.value`.

Implement a countdown that yields every positive integer and returns `"lift-off"`.

### Solution

In [6]:
def countdown(start: int) -> Iterator[int]:
    if start < 0:
        raise ValueError("start must be non-negative")

    while start > 0:
        yield start
        start -= 1

    return "lift-off"


timer = countdown(3)
assert [next(timer), next(timer), next(timer)] == [3, 2, 1]

try:
    next(timer)
except StopIteration as exc:
    completion_message = exc.value

assert completion_message == "lift-off"

## Problem 4 — Implement `exhaust_with_result`

Write a reusable helper that exhausts a generator and returns both all yielded values and the final return value.

Do not use a `for` loop, because it discards `StopIteration.value`.

### Solution

In [7]:
def exhaust_with_result(
    generator: Generator[T, Any, Any],
) -> tuple[list[T], Any]:
    yielded: list[T] = []

    while True:
        try:
            yielded.append(next(generator))
        except StopIteration as exc:
            return yielded, exc.value


values, result = exhaust_with_result(countdown(4))
assert values == [4, 3, 2, 1]
assert result == "lift-off"

## 2. Bidirectional generators: `send`, `throw`, and `close`

## Problem 5 — Running average with `send`

Build a generator that:

- is primed with `next`;
- receives numbers with `.send(number)`;
- yields the updated running average;
- rejects non-numeric input with `TypeError`.

### Solution

In [8]:
from numbers import Real

def running_average() -> Generator[float | None, Real, None]:
    total = 0.0
    count = 0
    incoming = yield None

    while True:
        if not isinstance(incoming, Real):
            raise TypeError("running_average accepts real numbers only")

        total += float(incoming)
        count += 1
        incoming = yield total / count


averager = running_average()
assert next(averager) is None
assert averager.send(10) == 10.0
assert averager.send(20) == 15.0
assert averager.send(-6) == 8.0
averager.close()
assert getgeneratorstate(averager) == "GEN_CLOSED"

## Problem 6 — Reset state with `throw`

Create a counter that normally increments, but resets to zero when the caller injects a custom `ResetCounter` exception with `.throw(...)`.

### Solution

In [9]:
class ResetCounter(Exception):
    """Signal that a counter should reset to zero."""


def controllable_counter() -> Generator[int, int | None, None]:
    value = 0

    while True:
        try:
            increment = yield value
        except ResetCounter:
            value = 0
        else:
            value += 1 if increment is None else increment


counter = controllable_counter()
assert next(counter) == 0
assert counter.send(None) == 1
assert counter.send(5) == 6
assert counter.throw(ResetCounter) == 0
assert counter.send(2) == 2
counter.close()

## Problem 7 — Guarantee cleanup on `close`

Write a generator that records `"opened"` before yielding and `"closed"` in `finally`. Verify that `.close()` while suspended executes the cleanup.

### Solution

In [10]:
cleanup_events: list[str] = []

def managed_sequence() -> Iterator[int]:
    cleanup_events.append("opened")
    try:
        yield 10
        yield 20
    finally:
        cleanup_events.append("closed")


managed = managed_sequence()
assert next(managed) == 10
assert cleanup_events == ["opened"]

managed.close()
assert cleanup_events == ["opened", "closed"]
assert getgeneratorstate(managed) == "GEN_CLOSED"

## Problem 8 — Explain `GeneratorExit`

When `.close()` is called, Python raises `GeneratorExit` at the suspension point. A well-behaved generator cleans up and stops; it must not yield a new value from its `GeneratorExit` handler.

### Solution

In [11]:
exit_events: list[str] = []

def generator_exit_observer() -> Iterator[str]:
    try:
        yield "ready"
    except GeneratorExit:
        exit_events.append("GeneratorExit observed")
        raise
    finally:
        exit_events.append("finally executed")


observer = generator_exit_observer()
assert next(observer) == "ready"
observer.close()

assert exit_events == [
    "GeneratorExit observed",
    "finally executed",
]

## 3. Delegation with `yield from`

## Problem 9 — Recursive lazy flattening

Flatten arbitrarily nested lists and tuples lazily. Treat strings and bytes as atomic values.

### Solution

In [12]:
def flatten(items: Iterable[object]) -> Iterator[object]:
    for item in items:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)
        else:
            yield item


nested = [1, (2, [3, (4,)]), "abc", b"xy", [], [5]]
assert list(flatten(nested)) == [1, 2, 3, 4, "abc", b"xy", 5]

## Problem 10 — Collect a subgenerator's return value

Create a child generator that yields `1..n` and returns their sum. Create a parent that delegates with `yield from`, receives the returned sum, and yields a summary record.

### Solution

In [13]:
def numbers_and_sum(limit: int) -> Generator[int, None, int]:
    total = 0
    for number in range(1, limit + 1):
        total += number
        yield number
    return total


def report_numbers(limit: int) -> Iterator[int | tuple[str, int]]:
    total = yield from numbers_and_sum(limit)
    yield ("sum", total)


assert list(report_numbers(5)) == [1, 2, 3, 4, 5, ("sum", 15)]

## Problem 11 — Forward `send` through `yield from`

Build a child accumulator and a parent that logs start/end events while delegating. The parent's `.send(...)` calls should reach the child.

### Solution

In [14]:
delegation_events: list[str] = []

def accumulator() -> Generator[int, int, None]:
    total = 0
    while True:
        incoming = yield total
        total += incoming


def delegated_accumulator() -> Generator[int, int, None]:
    delegation_events.append("delegation started")
    try:
        yield from accumulator()
    finally:
        delegation_events.append("delegation ended")


delegated = delegated_accumulator()
assert next(delegated) == 0
assert delegated.send(4) == 4
assert delegated.send(7) == 11
delegated.close()

assert delegation_events == [
    "delegation started",
    "delegation ended",
]

## Problem 12 — Forward exceptions through `yield from`

Create a child that catches `SkipCurrent`, yields a marker, and continues. Confirm that `.throw(SkipCurrent)` on the parent reaches the child.

### Solution

In [15]:
class SkipCurrent(Exception):
    pass


def resilient_child() -> Generator[str, None, None]:
    index = 0
    while index < 3:
        try:
            yield f"item:{index}"
            index += 1
        except SkipCurrent:
            yield f"skipped:{index}"
            index += 1


def resilient_parent() -> Generator[str, None, None]:
    yield "parent:start"
    yield from resilient_child()
    yield "parent:end"


resilient = resilient_parent()
assert next(resilient) == "parent:start"
assert next(resilient) == "item:0"
assert resilient.throw(SkipCurrent) == "skipped:0"
assert next(resilient) == "item:1"
assert next(resilient) == "item:2"
assert next(resilient) == "parent:end"

## 4. Lazy algorithms

## Problem 13 — Fixed-size chunking

Implement `chunked(iterable, size)` with lazy consumption. The final chunk may be shorter. Reject non-positive sizes.

### Solution

In [16]:
from itertools import islice

def chunked(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)

    while True:
        chunk = tuple(islice(iterator, size))
        if not chunk:
            return
        yield chunk


assert list(chunked(range(10), 4)) == [
    (0, 1, 2, 3),
    (4, 5, 6, 7),
    (8, 9),
]

## Problem 14 — Sliding windows

Implement a lazy `sliding_window(iterable, width)` that yields overlapping tuples.

### Solution

In [17]:
from collections import deque

def sliding_window(iterable: Iterable[T], width: int) -> Iterator[tuple[T, ...]]:
    if width <= 0:
        raise ValueError("width must be positive")

    iterator = iter(iterable)
    window: deque[T] = deque(islice(iterator, width), maxlen=width)

    if len(window) < width:
        return

    yield tuple(window)

    for item in iterator:
        window.append(item)
        yield tuple(window)


assert list(sliding_window([1, 2, 3, 4, 5], 3)) == [
    (1, 2, 3),
    (2, 3, 4),
    (3, 4, 5),
]
assert list(sliding_window([1, 2], 3)) == []

## Problem 15 — Merge two sorted streams

Merge two sorted iterables without materializing either one. Preserve duplicates. Target `O(n + m)` time and `O(1)` auxiliary space.

### Solution

In [18]:
_sentinel = object()

def merge_sorted(left: Iterable[T], right: Iterable[T]) -> Iterator[T]:
    left_iter = iter(left)
    right_iter = iter(right)

    left_value = next(left_iter, _sentinel)
    right_value = next(right_iter, _sentinel)

    while left_value is not _sentinel and right_value is not _sentinel:
        if left_value <= right_value:
            yield left_value
            left_value = next(left_iter, _sentinel)
        else:
            yield right_value
            right_value = next(right_iter, _sentinel)

    while left_value is not _sentinel:
        yield left_value
        left_value = next(left_iter, _sentinel)

    while right_value is not _sentinel:
        yield right_value
        right_value = next(right_iter, _sentinel)


assert list(merge_sorted([1, 3, 3, 8], [2, 3, 7, 9])) == [
    1, 2, 3, 3, 3, 7, 8, 9
]

## Problem 16 — Fair round-robin scheduling

Given several finite iterables, yield one item from each active iterator in turn. Remove an iterator only when exhausted.

### Solution

In [19]:
def round_robin(*iterables: Iterable[T]) -> Iterator[T]:
    active: deque[Iterator[T]] = deque(map(iter, iterables))

    while active:
        iterator = active.popleft()
        try:
            yield next(iterator)
        except StopIteration:
            continue
        active.append(iterator)


assert list(round_robin("ABC", [1, 2], ("x", "y", "z", "w"))) == [
    "A", 1, "x",
    "B", 2, "y",
    "C", "z",
    "w",
]

## Problem 17 — Lazy distinct values

Implement `unique_everseen`. The input is streamed, but note that the set of previous keys may still grow without bound.

### Solution

In [20]:
from collections.abc import Callable, Hashable

def unique_everseen(
    iterable: Iterable[T],
    key: Callable[[T], Hashable] | None = None,
) -> Iterator[T]:
    seen: set[Hashable] = set()

    for item in iterable:
        marker = item if key is None else key(item)
        if marker not in seen:
            seen.add(marker)
            yield item


assert list(unique_everseen([1, 1, 2, 1, 3, 2])) == [1, 2, 3]
assert list(unique_everseen(["A", "a", "B", "b"], key=str.lower)) == ["A", "B"]

## 5. Streaming pipelines

## Problem 18 — Build a composable log-processing pipeline

Lazily clean lines, parse `LEVEL|service|latency_ms`, skip malformed rows, retain slow records, and emit compact alert dictionaries.

### Solution

In [21]:
from dataclasses import dataclass

@dataclass(frozen=True)
class LogRecord:
    level: str
    service: str
    latency_ms: int


def cleaned_lines(lines: Iterable[str]) -> Iterator[str]:
    for line in lines:
        cleaned = line.strip()
        if cleaned and not cleaned.startswith("#"):
            yield cleaned


def parsed_records(lines: Iterable[str]) -> Iterator[LogRecord]:
    for line in lines:
        parts = line.split("|")
        if len(parts) != 3:
            continue

        level, service, latency_text = parts

        try:
            latency_ms = int(latency_text)
        except ValueError:
            continue

        yield LogRecord(level, service, latency_ms)


def slow_records(
    records: Iterable[LogRecord],
    threshold_ms: int,
) -> Iterator[LogRecord]:
    for record in records:
        if record.latency_ms >= threshold_ms:
            yield record


def alerts(records: Iterable[LogRecord]) -> Iterator[dict[str, object]]:
    for record in records:
        yield {
            "service": record.service,
            "severity": record.level,
            "latency_ms": record.latency_ms,
        }


raw_logs = [
    "INFO|catalog|80",
    "",
    "# generated by load test",
    "WARN|checkout|540",
    "bad record",
    "ERROR|payments|900",
    "INFO|search|not-a-number",
]

pipeline = alerts(
    slow_records(
        parsed_records(cleaned_lines(raw_logs)),
        threshold_ms=500,
    )
)

assert list(pipeline) == [
    {"service": "checkout", "severity": "WARN", "latency_ms": 540},
    {"service": "payments", "severity": "ERROR", "latency_ms": 900},
]

## Problem 19 — Add observability without breaking laziness

Implement `tap(iterable, observer)` that observes each pulled item and yields it unchanged.

### Solution

In [22]:
def tap(
    iterable: Iterable[T],
    observer: Callable[[T], None],
) -> Iterator[T]:
    for item in iterable:
        observer(item)
        yield item


observed: list[int] = []
stream = tap((number * number for number in range(4)), observed.append)

assert next(stream) == 0
assert observed == [0]

assert list(stream) == [1, 4, 9]
assert observed == [0, 1, 4, 9]

## Problem 20 — Batching coroutine with backpressure

Receive values via `.send(value)`. Yield a tuple whenever the buffer reaches `batch_size`; otherwise yield `None`. A sentinel flushes the final partial batch.

### Solution

In [23]:
FLUSH = object()

def batching_coroutine(
    batch_size: int,
) -> Generator[tuple[object, ...] | None, object, None]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    buffer: list[object] = []
    incoming = yield None

    while True:
        if incoming is FLUSH:
            outgoing = tuple(buffer) if buffer else None
            buffer.clear()
        else:
            buffer.append(incoming)
            outgoing = tuple(buffer) if len(buffer) == batch_size else None
            if outgoing is not None:
                buffer.clear()

        incoming = yield outgoing


batcher = batching_coroutine(3)
assert next(batcher) is None
assert batcher.send("a") is None
assert batcher.send("b") is None
assert batcher.send("c") == ("a", "b", "c")
assert batcher.send("d") is None
assert batcher.send(FLUSH) == ("d",)
batcher.close()

## 6. Introspection and debugging

## Problem 21 — Inspect suspended local variables

Use `.gi_frame.f_locals` to inspect a suspended generator's local variables. Treat this as a debugging technique, not a stable application API.

### Solution

In [24]:
def running_total(values: Iterable[int]) -> Iterator[int]:
    total = 0
    for index, value in enumerate(values):
        total += value
        yield total


totals = running_total([5, 7, 11])
assert next(totals) == 5

frame = totals.gi_frame
assert frame is not None

locals_snapshot = dict(frame.f_locals)
assert locals_snapshot["total"] == 5
assert locals_snapshot["index"] == 0
assert locals_snapshot["value"] == 5

totals.close()
assert totals.gi_frame is None

## Problem 22 — Detect generator re-entrancy

A generator cannot resume itself while already running. Demonstrate the resulting `ValueError` safely.

### Solution

In [25]:
reentrant_holder: dict[str, Iterator[str]] = {}

def reentrant_demo() -> Iterator[str]:
    try:
        next(reentrant_holder["generator"])
    except ValueError as exc:
        yield str(exc)


reentrant_holder["generator"] = reentrant_demo()
message = next(reentrant_holder["generator"])
assert "already executing" in message.lower()

## Problem 23 — Create a diagnostic snapshot

Report state, function name, last instruction offset, and visible locals when a frame exists.

### Solution

In [26]:
def generator_snapshot(generator: Iterator[object]) -> dict[str, object]:
    frame = generator.gi_frame
    code_object = generator.gi_code

    return {
        "state": getgeneratorstate(generator),
        "function": code_object.co_name,
        "instruction_offset": None if frame is None else frame.f_lasti,
        "locals": {} if frame is None else dict(frame.f_locals),
    }


diagnostic_generator = running_total([2, 4])
assert generator_snapshot(diagnostic_generator)["state"] == "GEN_CREATED"

assert next(diagnostic_generator) == 2
suspended_snapshot = generator_snapshot(diagnostic_generator)
assert suspended_snapshot["state"] == "GEN_SUSPENDED"
assert suspended_snapshot["locals"]["total"] == 2

diagnostic_generator.close()
closed_snapshot = generator_snapshot(diagnostic_generator)
assert closed_snapshot["state"] == "GEN_CLOSED"
assert closed_snapshot["locals"] == {}

## 7. Resource safety

## Problem 24 — Close an owned stream on early termination

Implement a line generator that owns a text stream and always closes it, even when the consumer stops early.

### Solution

In [27]:
from io import StringIO, TextIOBase

def owned_lines(stream: TextIOBase) -> Iterator[str]:
    try:
        for line in stream:
            yield line.rstrip("\n")
    finally:
        stream.close()


stream = StringIO("alpha\nbeta\ngamma\n")
line_generator = owned_lines(stream)

assert next(line_generator) == "alpha"
assert not stream.closed

line_generator.close()
assert stream.closed

## Problem 25 — Compose cleanup across delegated generators

Verify that closing a parent also closes its currently active delegated child.

### Solution

In [28]:
nested_cleanup: list[str] = []

def child_resource() -> Iterator[int]:
    try:
        yield 1
        yield 2
    finally:
        nested_cleanup.append("child closed")


def parent_resource() -> Iterator[int]:
    try:
        yield from child_resource()
    finally:
        nested_cleanup.append("parent closed")


parent = parent_resource()
assert next(parent) == 1
parent.close()

assert nested_cleanup == ["child closed", "parent closed"]

## 8. Testing patterns

## Problem 26 — Implement `take`

Return at most `count` items without over-consuming the iterator.

### Solution

In [29]:
def take(count: int, iterable: Iterable[T]) -> list[T]:
    if count < 0:
        raise ValueError("count must be non-negative")
    return list(islice(iterable, count))


source = iter(range(10))
assert take(3, source) == [0, 1, 2]
assert next(source) == 3

## Problem 27 — Test laziness explicitly

Create a source that records every pull. Confirm that requesting two items produces exactly two values.

### Solution

In [30]:
pulls: list[int] = []

def observable_source(limit: int) -> Iterator[int]:
    for number in range(limit):
        pulls.append(number)
        yield number


lazy_source = observable_source(100)
assert take(2, lazy_source) == [0, 1]
assert pulls == [0, 1]

## Problem 28 — Test exceptional behavior

Create `assert_raises` using only the standard library, then validate invalid arguments.

### Solution

In [31]:
from contextlib import contextmanager
from collections.abc import Iterator as ABCIterator

@contextmanager
def assert_raises(expected_exception: type[BaseException]) -> ABCIterator[None]:
    try:
        yield
    except expected_exception:
        return
    except BaseException as exc:
        raise AssertionError(
            f"expected {expected_exception.__name__}, "
            f"got {type(exc).__name__}"
        ) from exc
    raise AssertionError(f"{expected_exception.__name__} was not raised")


with assert_raises(ValueError):
    list(chunked([1, 2, 3], 0))

with assert_raises(ValueError):
    list(sliding_window([1, 2, 3], -1))

## 9. Performance and memory

## Problem 29 — Compare object sizes

Use `sys.getsizeof` to compare a materialized list with a generator expression. This does not measure the complete transitive object graph, but it illustrates lazy allocation.

### Solution

In [32]:
import sys

materialized = [number * number for number in range(100_000)]
lazy = (number * number for number in range(100_000))

list_object_bytes = sys.getsizeof(materialized)
generator_object_bytes = sys.getsizeof(lazy)

assert list_object_bytes > generator_object_bytes
{
    "list_object_bytes": list_object_bytes,
    "generator_object_bytes": generator_object_bytes,
}

{'list_object_bytes': 800984, 'generator_object_bytes': 200}

## Problem 30 — Sum a large conceptual stream without storing it

Compute the sum of squares from `0` through `999_999` lazily and verify it against the closed-form formula.

### Solution

In [33]:
n = 999_999
lazy_sum = sum(number * number for number in range(n + 1))
formula_sum = n * (n + 1) * (2 * n + 1) // 6

assert lazy_sum == formula_sum

## 10. Advanced control-flow challenges

## Problem 31 — Prunable depth-first traversal with `send`

Traverse a tree lazily. After each yielded node, the caller may send `True` to prune that node's children or `False`/`None` to continue.

### Solution

In [34]:
Tree = dict[str, list[str]]

def prunable_depth_first(
    tree: Tree,
    root: str,
) -> Generator[str, bool | None, None]:
    stack = [root]

    while stack:
        node = stack.pop()
        prune = yield node

        if not prune:
            children = tree.get(node, [])
            stack.extend(reversed(children))


tree = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F"],
    "D": [],
    "E": [],
    "F": [],
}

walker = prunable_depth_first(tree, "A")
assert next(walker) == "A"
assert walker.send(False) == "B"
assert walker.send(True) == "C"
assert walker.send(False) == "F"

try:
    walker.send(False)
except StopIteration:
    pass

## Problem 32 — Stateful Fibonacci stream with reset

Build an infinite Fibonacci generator. Sending an integer resets the state to `(0, reset_value)`. Sending `None` continues normally.

### Solution

In [35]:
def resettable_fibonacci() -> Generator[int, int | None, None]:
    previous, current = 0, 1

    while True:
        reset_value = yield previous

        if reset_value is None:
            previous, current = current, previous + current
        else:
            previous, current = 0, reset_value


fib = resettable_fibonacci()
assert next(fib) == 0
assert fib.send(None) == 1
assert fib.send(None) == 1
assert fib.send(None) == 2
assert fib.send(10) == 0
assert fib.send(None) == 10
assert fib.send(None) == 10
fib.close()

## Problem 33 — Streaming parser with line numbers

Parse `key=value` records lazily. Yield either an `"ok"` record or an `"error"` record. Ignore blank lines.

### Solution

In [36]:
def parse_assignments(
    lines: Iterable[str],
) -> Iterator[tuple[object, ...]]:
    for line_number, raw_line in enumerate(lines, start=1):
        line = raw_line.strip()

        if not line:
            continue

        if "=" not in line:
            yield ("error", line_number, raw_line)
            continue

        key, value = line.split("=", maxsplit=1)
        key = key.strip()
        value = value.strip()

        if not key:
            yield ("error", line_number, raw_line)
            continue

        yield ("ok", line_number, key, value)


assignment_lines = [
    "host=localhost",
    "",
    "invalid",
    " port = 5432 ",
    "=missing-key",
]

assert list(parse_assignments(assignment_lines)) == [
    ("ok", 1, "host", "localhost"),
    ("error", 3, "invalid"),
    ("ok", 4, "port", "5432"),
    ("error", 5, "=missing-key"),
]

## Problem 34 — K-way lazy merge

Use a heap to merge any number of sorted iterables. Preserve duplicates and keep only one look-ahead value from each active iterator.

### Solution

In [37]:
import heapq

def merge_many_sorted(*iterables: Iterable[T]) -> Iterator[T]:
    heap: list[tuple[T, int, Iterator[T]]] = []

    for source_index, iterable in enumerate(iterables):
        iterator = iter(iterable)
        first = next(iterator, _sentinel)
        if first is not _sentinel:
            heapq.heappush(heap, (first, source_index, iterator))

    while heap:
        value, source_index, iterator = heapq.heappop(heap)
        yield value

        following = next(iterator, _sentinel)
        if following is not _sentinel:
            heapq.heappush(heap, (following, source_index, iterator))


assert list(
    merge_many_sorted(
        [1, 10, 20],
        [2, 3, 30],
        [],
        [0, 4, 40],
    )
) == [0, 1, 2, 3, 4, 10, 20, 30, 40]

## Problem 35 — Retry wrapper for a deterministic iterator factory

An iterator may fail during iteration. Retry with a fresh iterator, skip the already emitted prefix, and stop after `max_retries`.

This pattern is safe only when recreating the iterator is deterministic and replayable.

### Solution

In [38]:
class TransientError(RuntimeError):
    pass


def retrying_stream(
    factory: Callable[[], Iterator[T]],
    max_retries: int,
) -> Iterator[T]:
    if max_retries < 0:
        raise ValueError("max_retries must be non-negative")

    emitted_count = 0
    attempts = 0

    while True:
        iterator = factory()

        for _ in range(emitted_count):
            next(iterator)

        try:
            while True:
                item = next(iterator)
                emitted_count += 1
                yield item
        except StopIteration:
            return
        except TransientError:
            attempts += 1
            if attempts > max_retries:
                raise


factory_calls = 0

def flaky_factory() -> Iterator[int]:
    global factory_calls
    factory_calls += 1

    for number in range(5):
        if factory_calls == 1 and number == 2:
            raise TransientError("temporary failure")
        yield number


assert list(retrying_stream(flaky_factory, max_retries=1)) == [0, 1, 2, 3, 4]
assert factory_calls == 2

## 11. Common pitfalls and corrected patterns

### Pitfall A — A generator is one-shot

Once exhausted, iterating it again yields nothing. Store a generator factory when repeatability is required.

In [39]:
one_shot = (number for number in range(3))
assert list(one_shot) == [0, 1, 2]
assert list(one_shot) == []

repeatable = lambda: (number for number in range(3))
assert list(repeatable()) == [0, 1, 2]
assert list(repeatable()) == [0, 1, 2]

### Pitfall B — Membership tests consume generators

`value in generator` advances the generator up to the match or exhaustion.

In [40]:
membership_stream = iter([10, 20, 30, 40])

assert 30 in membership_stream
assert list(membership_stream) == [40]

### Pitfall C — Do not raise `StopIteration` manually inside a generator

Use `return` to terminate a generator.

In [41]:
def correct_termination() -> Iterator[int]:
    yield 1
    return

assert list(correct_termination()) == [1]

### Pitfall D — `itertools.tee` may buffer heavily

Duplicating an iterator can require large internal buffering when one branch advances far ahead of another.

In [42]:
from itertools import tee

original = (number for number in range(5))
branch_a, branch_b = tee(original, 2)

assert next(branch_a) == 0
assert list(branch_b) == [0, 1, 2, 3, 4]
assert list(branch_a) == [1, 2, 3, 4]

## 12. Capstone — Streaming transaction analytics

Build a fully lazy transaction pipeline that:

1. validates required fields;
2. converts amount to `float`;
3. rejects non-positive amounts;
4. filters for a selected region;
5. emits rolling windows of three accepted transactions;
6. returns window summaries with count, total, and average.

Malformed records are routed to an error callback.

### Solution

In [43]:
from collections.abc import Mapping

@dataclass(frozen=True)
class Transaction:
    transaction_id: str
    region: str
    amount: float


def validated_transactions(
    records: Iterable[Mapping[str, object]],
    on_error: Callable[[Mapping[str, object], str], None],
) -> Iterator[Transaction]:
    required = {"id", "region", "amount"}

    for record in records:
        missing = required.difference(record)
        if missing:
            on_error(record, f"missing fields: {sorted(missing)}")
            continue

        try:
            amount = float(record["amount"])
        except (TypeError, ValueError):
            on_error(record, "amount is not numeric")
            continue

        if amount <= 0:
            on_error(record, "amount must be positive")
            continue

        yield Transaction(
            transaction_id=str(record["id"]),
            region=str(record["region"]),
            amount=amount,
        )


def region_only(
    transactions: Iterable[Transaction],
    region: str,
) -> Iterator[Transaction]:
    for transaction in transactions:
        if transaction.region == region:
            yield transaction


def summarize_transaction_windows(
    windows: Iterable[tuple[Transaction, ...]],
) -> Iterator[dict[str, object]]:
    for window in windows:
        total = sum(transaction.amount for transaction in window)
        yield {
            "ids": tuple(transaction.transaction_id for transaction in window),
            "count": len(window),
            "total": round(total, 2),
            "average": round(total / len(window), 2),
        }


raw_transactions = [
    {"id": "t1", "region": "EU", "amount": "10.50"},
    {"id": "t2", "region": "US", "amount": 99},
    {"id": "t3", "region": "EU", "amount": 20},
    {"id": "bad1", "region": "EU"},
    {"id": "t4", "region": "EU", "amount": 30},
    {"id": "bad2", "region": "EU", "amount": -4},
    {"id": "t5", "region": "EU", "amount": 40},
]

transaction_errors: list[tuple[Mapping[str, object], str]] = []

def record_transaction_error(
    record: Mapping[str, object],
    reason: str,
) -> None:
    transaction_errors.append((record, reason))


transaction_pipeline = summarize_transaction_windows(
    sliding_window(
        region_only(
            validated_transactions(
                raw_transactions,
                record_transaction_error,
            ),
            region="EU",
        ),
        width=3,
    )
)

transaction_summaries = list(transaction_pipeline)

assert transaction_summaries == [
    {
        "ids": ("t1", "t3", "t4"),
        "count": 3,
        "total": 60.5,
        "average": 20.17,
    },
    {
        "ids": ("t3", "t4", "t5"),
        "count": 3,
        "total": 90.0,
        "average": 30.0,
    },
]

assert [reason for _, reason in transaction_errors] == [
    "missing fields: ['amount']",
    "amount must be positive",
]

## Final review questions

1. Why is a generator `GEN_CREATED` before its first `next()`?
2. At what exact point does execution suspend around `yield expression`?
3. Why does a `for` loop hide `StopIteration.value`?
4. What may be sent into a newly created generator?
5. How do `.throw()` and `.close()` resume a suspended generator?
6. Why is `yield from` better than a manual forwarding loop?
7. Which lazy algorithms still need potentially unbounded memory?
8. What happens when a generator attempts to resume itself?
9. How should resource-owning generators handle early termination?
10. How can a test prove that a pipeline is genuinely lazy?

## Summary

Generators are suspended execution frames, not merely compact list builders. Their power comes from combining lifecycle states, lazy iteration, bidirectional communication, exception injection, deterministic cleanup, delegation, and composable streaming transformations.